In [ ]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt

plt.style.use('default')  # resets to light theme


In [ ]:
import xarray as xr
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import BaggingRegressor
from catboost import CatBoostRegressor

from highres_ta.evaluation import scoring, make_prediction_df,  build_groupers
from highres_ta.preprocessing import compute_n_coords, load_config, load_data, preprocess_data, get_train_test_data

# Load train, test data

In [ ]:
config = load_config("./training_iterations_config.yaml")
print(config)

In [ ]:
df_raw = load_data()

df = preprocess_data(df_raw, config)

train_x, train_y, test_x, test_y = get_train_test_data(df, config)


In [ ]:
train_x

In [ ]:
train_y

# Fit models

In [ ]:
import highres_ta.estimators as models

In [ ]:
print(config.bagged_estimator_params)


# bagged_model = models.BaggingCatBoostResidualRegressor(
#     n_estimators=20, 
#     polynomial_degree=1,
#     max_samples=0.2,
#     loss_function="MAE",
#     linear_features=config.linear_features,
#     n_jobs=8,
# )

bagged_model = models.BaggingCatBoostResidualRegressor(**config.bagged_estimator_params)


In [ ]:
bagged_model.fit(train_x, train_y)
train_yhat = bagged_model.predict(train_x)
train_predictions_df = make_prediction_df(y_pred = train_yhat, y_true=train_y) 
train_scores = scoring(train_predictions_df)

In [ ]:
# train_predictions_df = make_prediction_df(y_pred = train_yhat, y_true=train_y) 
# train_predictions_df
# train_scores = scoring(train_predictions_df)

In [ ]:
# grouper = build_groupers(train_predictions_df, time_freq='year',salinity_group=True)
# train_predictions_df_grouped = train_predictions_df.groupby(grouper)
# train_predictions_df_grouped

In [ ]:
# time_index = train_predictions_df.index.get_level_values('time')
# yearly_train_scores = grouped_scoring(train_predictions_df)
# yearly_train_scores




In [ ]:
print(bagged_model.feature_names_in_)

# Test scores

In [ ]:
# the prediction flow:
# 1) get the baseline prediction from the linear model
# 2) get the residual prediction from the bagged booster
# 3) sum them to get the final prediction
test_yhat = bagged_model.predict(test_x)
test_predictions_df = make_prediction_df(test_yhat, test_y)
test_scores = scoring(test_predictions_df)

# Load inference data (single timestep)

In [ ]:
url = "https://data.up.ethz.ch/shared/OceanSODA-ETHZv2/.inference_for_gregor2024/data_8daily_25km_v01.zarr/"
ds = xr.open_zarr(url, consolidated=True, group='2004')

ds["bottomdepth"] = xr.open_dataarray("../data/bathymetry_etopo2022_25km.nc")
rename = dict(
    sss="salinity",
    sst="temperature",
    ssh="ssh_adt",
    ssh_anom="ssh_sla",
    mld="mld_dens_soda",
    chl_filled="chl_globcolour",
    bottomdepth="bottomdepth",
)
ds = ds.rename(rename)[list(rename.values())]
ds = ds.isel(time=33, drop=True)
df = ds.to_dataframe()
coords = df.index.to_frame()
df['ncoord_a'], df['ncoord_b'], df['ncoord_c'] = compute_n_coords(coords['lat'], coords['lon'])
df['depth'] = 0

coords = df.index.to_frame()
n_coords = compute_n_coords(coords["lat"], coords["lon"])
df = df.join(n_coords)
pred_X = df.dropna()[train_x.columns]

In [ ]:
df

# Create mapped estimates

In [ ]:
import numpy as np
full_results = np.vstack((bagged_model.predict(pred_X, return_std=True),
bagged_model.predict_linear(pred_X, return_std=True),
bagged_model.predict_catboost(pred_X, return_std=True),))

In [ ]:
pred_y = pd.DataFrame(
    data=full_results.T,
    columns=["mean_prediction", "std_prediction", "mean_linear", "std_linear", "mean_catboost", "std_catboost"],
    index=pred_X.index,
).to_xarray().astype("float32").chunk()
pred_y["std_combined"] = ((pred_y.std_catboost**2 + pred_y.std_linear**2)**0.5).chunk()

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(3, 2, figsize=(12, 8), squeeze=False, sharex=True, sharey=True, constrained_layout=True)
img1 = pred_y.mean_prediction.plot.imshow(vmin=2200, vmax=2450, ax=axs[0, 0], cmap="Spectral_r")
img2 = pred_y.std_combined.plot.imshow(vmin=0, vmax=15, ax=axs[0, 1])
img3 = pred_y.mean_linear.plot.imshow(vmin=2200, vmax=2450, ax=axs[1, 0], cmap="Spectral_r")
img4 = pred_y.std_linear.plot.imshow(vmin=0, vmax=15, ax=axs[1, 1])
img5 = pred_y.mean_catboost.plot.imshow(robust=True, ax=axs[2, 0])
img6 = pred_y.std_catboost.plot.imshow(vmin=0, vmax=15, ax=axs[2, 1])

[ax.set_ylabel('') for ax in axs.flat]
[ax.set_xlabel('') for ax in axs.flat]

# img3.get_clim()
# img1.set_clim(img3.get_clim())

img1.colorbar.set_label("Total Alkalinity (µmol/kg)")
img2.colorbar.set_label("∆ Total Alkalinity (µmol/kg)")
img3.colorbar.set_label("Total Alkalinity (µmol/kg)")
img4.colorbar.set_label("σ Residual (µmol/kg)")

text_props = dict(fontsize=14, fontweight='bold', color='black', loc='left', va='top')
axs = axs.flatten()
axs[0].set_title(" Final prediction", **text_props)
axs[1].set_title(" Combined σ", **text_props)
axs[2].set_title(" Linear baseline", **text_props)
axs[3].set_title(" Linear σ", **text_props)
axs[4].set_title(" CatBoost residual", **text_props)
axs[5].set_title(" Catboost σ", **text_props)

for ax in axs.flatten():
    ax.set_xlabel("")
    ax.set_ylabel("Latitude")
    

In [ ]:
pd.Series(bagged_model.estimators_[0].boosting_model_.get_feature_importance(), index=train_x.columns).pipe(lambda s: s / s.sum()).sort_values(ascending=True).round(2).plot.barh()